<a href="https://colab.research.google.com/github/Shineii86/MoeStickerBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&customColorList=12,14,20,24,27&height=200&section=header&text=Moe%20Sticker%20Bot&fontSize=60&fontColor=ffffff&animation=fadeIn&fontAlignY=35&desc=Telegram%20Sticker%20Bot%20in%20Google%20Colab&descSize=18&descAlignY=52" alt="Moe Sticker Bot">
  <h1>✨ Moe Sticker Bot — Ultimate Colab Edition (WebApp Ready)</h1>
  <p><b>Convert, edit, and create Telegram stickers with ease. Fully self-contained in Colab.</b></p>
</div>

---

<div align="center">

| 🎬 Animated Stickers | ✂️ Crop & Resize | 🔤 Text & Emoji | 🌐 LINE/Kakao Import |
|:---:|:---:|:---:|:---:|
| Convert videos to WebM | Precision cropping tools | Add custom captions | Decrypt and import packs |

</div>

---

### 🚀 Quick Start Guide

1.  🔑 **Get a Bot Token** from [@BotFather](https://t.me/botfather) on Telegram.
2.  ⚙️ **Enter the token** in the configuration section below.
3.  ▶️ **Run all cells** (Runtime → Run all).
4.  💬 **Open Telegram** and start chatting with your bot!

> 💡 **Pro Tip**: Keep this Colab tab open. Free sessions last ~90 minutes idle. Upgrade to Colab Pro for 24h runtime.

---

In [ ]:
#@title 📦 0. Install tqdm (Progress Bars)
!pip install -q tqdm
from tqdm.notebook import tqdm
import time
print('\033[92m✅ tqdm ready!\033[0m')

In [ ]:
#@title 📦 1. Install System Dependencies (with Progress)

import subprocess, sys, os, urllib.request
from tqdm.notebook import tqdm

# ANSI color helpers (built right in)
G = '\033[92m'; R = '\033[91m'; Y = '\033[93m'; B = '\033[94m'; C = '\033[96m'; W = '\033[0m'

print(f"{B}{'='*60}{W}")
print(f"{B}  📦 INSTALLING SYSTEM DEPENDENCIES{W}")
print(f"{B}{'='*60}{W}\n")

steps = [
    ("Updating package lists", "apt-get update -qq"),
    ("Installing ImageMagick, ffmpeg, exiv2, etc.", "apt-get install -y -qq imagemagick libarchive-tools ffmpeg curl gifsicle python3 exiv2"),
]

for desc, cmd in tqdm(steps, desc="System Setup", unit="step"):
    tqdm.write(f"{C}⏳ {desc}...{W}")
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Download Go with progress bar
url = "https://go.dev/dl/go1.21.5.linux-amd64.tar.gz"
filename = "go1.21.5.linux-amd64.tar.gz"
print(f"{C}⏳ Downloading Go...{W}")
with tqdm(unit='B', unit_scale=True, unit_divisor=1024, miniters=1, desc="Go Download") as t:
    urllib.request.urlretrieve(url, filename, reporthook=lambda b, bsize, tsize: t.update(b*bsize - t.n))

print(f"{C}⏳ Extracting Go...{W}")
!tar -C /usr/local -xzf go1.21.5.linux-amd64.tar.gz

os.environ['PATH'] += ":/usr/local/go/bin"
os.environ['GOPATH'] = "/root/go"
os.environ['GO111MODULE'] = "on"
!mkdir -p $GOPATH

print(f"\n{G}✅ All system dependencies installed!{W}")
print(f"{W}Installed Versions:")
print(f"  {G}• Go:{W}        {subprocess.getoutput('go version').split()[2]}")
print(f"  {G}• exiv2:{W}     {subprocess.getoutput('exiv2 --version | head -1').split()[-1]}")
print(f"  {G}• ffmpeg:{W}    {subprocess.getoutput('ffmpeg -version | head -1').split()[2]}")
print(f"  {G}• ImageMagick:{W} {subprocess.getoutput('convert --version | head -1').split()[2]}")


In [ ]:
#@title 🐍 2. Install Python Helper Scripts

print(f"{B}{'='*60}{W}")
print(f"{B}  🐍 INSTALLING PYTHON HELPERS{W}")
print(f"{B}{'='*60}{W}\n")

helpers = [
    ("msb_emoji.py", "Emoji processing"),
    ("msb_kakao_decrypt.py", "KakaoTalk decryption"),
    ("msb_rlottie.py", "Lottie animation support")
]

for filename, desc in tqdm(helpers, desc="Downloading Helpers", unit="file"):
    tqdm.write(f"{C}⏳ {desc}...{W}")
    !wget -q https://raw.githubusercontent.com/Shineii86/MoeStickersBot/master/tools/{filename} -O /usr/local/bin/{filename}
    !chmod +x /usr/local/bin/{filename}

print(f"\n{G}✅ All Python helpers installed to /usr/local/bin/{W}")


In [ ]:
#@title 🔨 3. Build MoeStickersBot from Source

print(f"{B}{'='*60}{W}")
print(f"{B}  🔨 BUILDING MoeStickersBot{W}")
print(f"{B}{'='*60}{W}\n")

!rm -rf MoeStickersBot

with tqdm(total=100, desc="Cloning repository", unit="%") as pbar:
    !git clone --depth 1 --progress https://github.com/Shineii86/MoeStickersBot.git 2>&1 | while read line; do echo $line; done
    pbar.update(100)

%cd MoeStickersBot

tqdm.write(f"{C}⏳ Downloading Go modules...{W}")
!go mod download

tqdm.write(f"{C}⏳ Compiling bot (this may take a minute)...{W}")
!go build -o MoeStickersBot cmd/MoeStickersBot/main.go

if os.path.exists("MoeStickersBot"):
    size = os.path.getsize("MoeStickersBot") / (1024 * 1024)
    print(f"\n{G}✅ Build successful! Binary ready ({size:.1f} MB){W}")
else:
    print(f"\n{R}❌ Build failed. Check output above.{W}")


In [ ]:
#@title ⚙️ 4. Bot Configuration

print(f"{B}{'='*60}{W}")
print(f"{B}  ⚙️ CONFIGURATION{W}")
print(f"{B}{'='*60}{W}\n")

# ========== REQUIRED ==========
BOT_TOKEN = ""  #@param {type:"string"}

# ========== DATABASE (OPTIONAL) ==========
ENABLE_DB = False  #@param {type:"boolean"}
DB_ADDR = "localhost:3306"  #@param {type:"string"}
DB_USER = "moe_bot"  #@param {type:"string"}
DB_PASS = ""  #@param {type:"string"}
DB_NAME = "moe_sticker_bot"  #@param {type:"string"}

# ========== WEBAPP (OPTIONAL - Requires ngrok) ==========
ENABLE_WEBAPP = False  #@param {type:"boolean"}
WEBAPP_PORT = 8080  #@param {type:"integer"}
NGROK_AUTHTOKEN = ""  #@param {type:"string"}
# Get your free ngrok auth token at https://dashboard.ngrok.com/auth

# ========== GENERAL ==========
DATA_DIR = "moe_sticker_bot_data"  #@param {type:"string"}
LOG_LEVEL = "info"  #@param ["debug", "info", "warn", "error"]
HTTP_PROXY = ""  #@param {type:"string"}

if BOT_TOKEN:
    masked = BOT_TOKEN[:8] + "..." + BOT_TOKEN[-4:]
    print(f"{G}✅ Bot Token:{W} {masked}")
else:
    print(f"{Y}⚠️  Please enter your Bot Token above.{W}")

if ENABLE_WEBAPP and not NGROK_AUTHTOKEN:
    print(f"{Y}⚠️  WebApp is enabled but no ngrok auth token provided. WebApp will be disabled.{W}")
    ENABLE_WEBAPP = False


In [ ]:
#@title 🌐 4.5 Setup ngrok Tunnel (if WebApp enabled)

import subprocess, json, requests, time, sys

WEBAPP_URL = ""
ngrok_process = None

if ENABLE_WEBAPP:
    print(f"{B}{'='*60}{W}")
    print(f"{B}  🌐 SETTING UP NGROK TUNNEL FOR WEBAPP{W}")
    print(f"{B}{'='*60}{W}\n")

    # Download ngrok if not present
    if not os.path.exists("./ngrok"):
        print(f"{C}⏳ Downloading ngrok...{W}")
        !wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
        !tar -xzf ngrok-v3-stable-linux-amd64.tgz
        !chmod +x ngrok

    # Authenticate
    !./ngrok config add-authtoken {NGROK_AUTHTOKEN}

    # Kill any previous ngrok instances
    !pkill -f ngrok || true

    # Start ngrok in background
    print(f"{C}⏳ Starting ngrok tunnel on port {WEBAPP_PORT}...{W}")
    ngrok_process = subprocess.Popen(
        ["./ngrok", "http", str(WEBAPP_PORT), "--log", "stdout"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    # Wait for tunnel to be ready and get public URL
    time.sleep(3)
    max_retries = 10
    for i in range(max_retries):
        try:
            resp = requests.get("http://127.0.0.1:4040/api/tunnels")
            if resp.status_code == 200:
                tunnels = resp.json()['tunnels']
                if tunnels:
                    WEBAPP_URL = tunnels[0]['public_url']
                    print(f"{G}✅ ngrok tunnel established!{W}")
                    print(f"{G}   Public URL: {WEBAPP_URL}{W}")
                    break
        except:
            pass
        time.sleep(1)
    else:
        print(f"{R}❌ Failed to retrieve ngrok URL. WebApp will be disabled.{W}")
        ENABLE_WEBAPP = False
else:
    print(f"{Y}ℹ️  WebApp not enabled. Skipping ngrok setup.{W}")


In [ ]:
#@title 🚀 5. Launch Bot (Background)

import subprocess, sys, time

print(f"{B}{'='*60}{W}")
print(f"{B}  🚀 LAUNCHING MoeStickersBot{W}")
print(f"{B}{'='*60}{W}\n")

if not BOT_TOKEN:
    print(f"{R}❌ No Bot Token provided. Please enter it in the Configuration cell.{W}")
    sys.exit(1)

cmd = [
    "./MoeStickersBot",
    f"--bot_token={BOT_TOKEN}",
    f"--log_level={LOG_LEVEL}",
    f"--data_dir={DATA_DIR}"
]

if ENABLE_DB and DB_ADDR:
    cmd.extend([
        f"--db_addr={DB_ADDR}",
        f"--db_user={DB_USER}",
        f"--db_pass={DB_PASS}",
        f"--db_name={DB_NAME}"
    ])

if ENABLE_WEBAPP and WEBAPP_URL:
    cmd.append(f"--webapp_url={WEBAPP_URL}")
    cmd.append(f"--webapp_listen_addr=0.0.0.0:{WEBAPP_PORT}")

if HTTP_PROXY:
    cmd.append(f"--http_proxy={HTTP_PROXY}")

print(f"{C}🔧 Command: {' '.join(cmd).replace(BOT_TOKEN, '[SECRET]')}{W}\n")

log_stdout = open("bot_stdout.log", "w")
log_stderr = open("bot_stderr.log", "w")
process = subprocess.Popen(cmd, stdout=log_stdout, stderr=log_stderr)

time.sleep(5)
if process.poll() is None:
    print(f"{G}✅ Bot is RUNNING!{W}")
    print(f"{G}📱 Go to Telegram and send /start to your bot!{W}")
    if ENABLE_WEBAPP and WEBAPP_URL:
        print(f"{G}🌐 WebApp is available at: {WEBAPP_URL}{W}")
    print(f"{G}📋 Logs: bot_stdout.log, bot_stderr.log{W}")
else:
    print(f"{R}❌ Bot exited. Check stderr log:{W}")
    !cat bot_stderr.log


In [ ]:
#@title 📜 View Live Logs
LOG_TYPE = "stderr"  #@param ["stdout", "stderr"]
LINES = 30  #@param {type:"slider", min:10, max:100, step:10}
log_file = f"bot_{LOG_TYPE}.log"
print(f"{C}📄 Last {LINES} lines of {log_file}:{W}\n")
!tail -n {LINES} {log_file}


In [ ]:
#@title 🛑 Stop Bot & ngrok
!pkill -f MoeStickersBot && echo "🛑 Bot stopped." || echo "ℹ️ No bot running."
!pkill -f ngrok && echo "🛑 ngrok stopped." || echo "ℹ️ No ngrok running."


---
<div align="center">
  <img src="https://capsule-render.vercel.app/api?type=waving&color=gradient&customColorList=12,14,20,24,27&height=100&section=footer" width="100%">
  <p>Made with ❤️ for the sticker community</p>
</div>